In [2]:
import pandas as pd
from rich.console import Console
from rich.table import Table
from rich.box import ROUNDED
from rich.panel import Panel
from rich import box

In [3]:
# load dataset
data = pd.read_csv('Dataset/dataset_clean.csv')

In [13]:
COLOR_DARK_NAVY = "#333F6E"
COLOR_BLUE_GREY = "#869EAF"
COLOR_BEIGE = "#DFD097"
COLOR_CREAM = "#CECEAB"

HTML_OUTPUT_PATH = "Report/hr_summary_report.html"


# record_console=True lets us export everything printed to HTML afterwards
console = Console(record=True)
 
# ---- Age statistics panel ----
age_stats = data['age'].agg(['min', 'mean', 'max'])
age_stats['mean'] = age_stats['mean'].round(1)
 
age_table = Table(
    title="Age Statistics",
    box=box.SIMPLE,
    header_style=f"bold {COLOR_DARK_NAVY}"
)
 
age_table.add_column("Measure", style=COLOR_BLUE_GREY)
age_table.add_column("Value", style=COLOR_BEIGE, justify="right")
 
age_table.add_row("Minimum", f"{age_stats['min']} years")
age_table.add_row("Mean", f"{age_stats['mean']} years")
age_table.add_row("Maximum", f"{age_stats['max']} years")
 
# ---- Gender distribution table ----
gender_counts = data['gender'].value_counts()
gender_pcts = data['gender'].value_counts(normalize=True) * 100
 
gender_table = Table(
    title="Gender Distribution",
    box=box.SIMPLE,
    header_style=f"bold {COLOR_DARK_NAVY}"
)
 
gender_table.add_column("Gender", style=COLOR_BLUE_GREY)
gender_table.add_column("Count", style=COLOR_BEIGE, justify="right")
gender_table.add_column("Percentage", style=COLOR_CREAM, justify="right")
 
for gender, count in gender_counts.items():
    gender_table.add_row(gender, f"{count:,}", f"{gender_pcts[gender]:.1f}%")
 
# ---- Role distribution table ----
role_counts = data['role'].value_counts()
role_pcts = data['role'].value_counts(normalize=True) * 100
 
role_table = Table(
    title="Role Distribution",
    box=box.SIMPLE,
    header_style=f"bold {COLOR_DARK_NAVY}"
)
 
role_table.add_column("Role", style=COLOR_BLUE_GREY)
role_table.add_column("Count", style=COLOR_BEIGE, justify="right")
role_table.add_column("Percentage", style=COLOR_CREAM, justify="right")
 
for role, count in role_counts.head(10).items():  # Show top 10 roles
    role_table.add_row(role, f"{count:,}", f"{role_pcts[role]:.1f}%")
 
# ---- Display all tables ----
console.print(age_table)
console.print(gender_table)
console.print(role_table)
 
# ---- Summary panel ----
console.print(Panel(
    f"[bold {COLOR_CREAM}]Total Employees:[/bold {COLOR_CREAM}] {len(data):,}\n"
    f"[bold {COLOR_CREAM}]Age Range:[/bold {COLOR_CREAM}] {age_stats['min']} - {age_stats['max']} years\n"
    f"[bold {COLOR_CREAM}]Most Common Gender:[/bold {COLOR_CREAM}] {gender_counts.index[0]}\n"
    f"[bold {COLOR_CREAM}]Most Common Role:[/bold {COLOR_CREAM}] {role_counts.index[0]} ({role_pcts.iloc[0]:.1f}%)",
    title="Summary",
    border_style=COLOR_BEIGE
))
 
# ---- Export to HTML with black background + custom theme ----
console.save_html(
    HTML_OUTPUT_PATH,
    theme=None,  # we override styling manually below for full control
    inline_styles=True
)
 
with open(HTML_OUTPUT_PATH, "r", encoding="utf-8") as f:
    html_content = f.read()

html_content = html_content.replace(
    "body {\n    color: #000000;\n    background-color: #ffffff;\n}",
    f"body {{\n    color: {COLOR_CREAM};\n    background-color: #000000;\n}}"
)
 
# Force black background on <body> (and the pre/code block rich wraps output in)
html_content = html_content.replace(
    "<body>",
    f'<body style="background-color:#000000; margin:0; padding:20px;">'
)
 
with open(HTML_OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(html_content)
 
print(f"\nSaved HTML report ")
 

     Age Statistics     
                        
  Measure        Value  
 ────────────────────── 
  Minimum   22.0 years  
  Mean      41.0 years  
  Maximum   60.0 years 

        Gender Distribution         
                                    
  Gender        Count   Percentage  
 ────────────────────────────────── 
  Male         11,900        47.6%  
  Female       11,878        47.5%  
  Non-binary    1,222         4.9% 

        Role Distribution         
                                  
  Role        Count   Percentage  
 ──────────────────────────────── 
  Support     5,912        23.6%  
  Developer   4,588        18.4%  
  Sales       4,481        17.9%  
  Manager     4,164        16.7%  
  HR          3,795        15.2%  
  Executive   2,060         8.2% 

╭──────────────────────────────────────────────────── Summary ────────────────────────────────────────────────────╮
│ Total Employees: 25,000                                                                                         │
│ Age Range: 22.0 - 60.0 years                                                                                    │
│ Most Common Gender: Male                                                                                        │
│ Most Common Role: Support (23.6%)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Saved HTML report 
